### Day5. 다중 선형 회귀

**Advertising 데이터셋**
- TV : TV 광고비(천 달러 단위)
- radio : 라디오 광고비
- newspaper : 신문 광고비
- sales : 제품 판매량 (단위: 천 개)
- 200개 데이터

In [2]:
# 데이터 이해
import pandas as pd
path = "https://www.statlearning.com/s/"
df = pd.read_csv(path + "Advertising.csv")
print(df.head(3))
print(df.info())

   Unnamed: 0     TV  radio  newspaper  sales
0           1  230.1   37.8       69.2   22.1
1           2   44.5   39.3       45.1   10.4
2           3   17.2   45.9       69.3    9.3
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  200 non-null    int64  
 1   TV          200 non-null    float64
 2   radio       200 non-null    float64
 3   newspaper   200 non-null    float64
 4   sales       200 non-null    float64
dtypes: float64(4), int64(1)
memory usage: 7.9 KB
None


다음과 같은 다중선형회귀 모형을 사용한 회귀모델을 만들고 결과를 확인합니다.
- Advertising.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 합니다.
- 종속변수 : sales
- 독립변수 : sales를 제외한 모든 변수
- 처음부터 순서대로 150개 데이터를 train, 나머지 50개를 test로 사용합니다.
- 모델 생성시 train 데이터를 사용합니다.

In [3]:
from statsmodels.api import OLS
pd.options.display.float_format = '{:.3f}'.format

train = df.iloc[:150]
test = df.iloc[150:]

formula = 'sales ~ ' + ' + '.join(df.columns[1:-1])
model = OLS.from_formula(formula, train).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.896
Model:                            OLS   Adj. R-squared:                  0.894
Method:                 Least Squares   F-statistic:                     418.2
Date:                Sat, 08 Nov 2025   Prob (F-statistic):           1.90e-71
Time:                        16:37:04   Log-Likelihood:                -291.75
No. Observations:                 150   AIC:                             591.5
Df Residuals:                     146   BIC:                             603.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.0298      0.371      8.172      0.0

In [4]:
#5-2) 위에서 생성한 모델에서 적합된 모형 결정계수를 구해 반올림하여 소수점 아래 3자리까지 출력한다.
print(round(model.rsquared, 3))

0.896


In [5]:
#5-3) 위에서 생성한 모델에서 통계적으로 유의미한 변수는 몇 개인가?
# 유의수준 : 5%
print(model.pvalues[1:]<=0.05)
print(sum(model.pvalues[1:]<=0.05))

TV            True
radio         True
newspaper    False
dtype: bool
2


In [6]:
#5-4) 위의 모델에서 통계적으로 유의한 변수들과 TV와 Radio의 교호작용항을 사용하여
# 새롭게 모델링하여 model2를 생성한다.
formula2 = 'sales ~ TV + radio + TV:radio'
model2 = OLS.from_formula(formula2, train).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.967
Model:                            OLS   Adj. R-squared:                  0.966
Method:                 Least Squares   F-statistic:                     1408.
Date:                Sat, 08 Nov 2025   Prob (F-statistic):          1.65e-107
Time:                        16:37:09   Log-Likelihood:                -206.40
No. Observations:                 150   AIC:                             420.8
Df Residuals:                     146   BIC:                             432.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      6.9633      0.300     23.232      0.0

In [8]:
#5-5) 다음의 값을 사용하여 예측값을 구해, 반올림하여 소수점아래 4자리까지 출력한다.
# TV = 170.5, radio=13.2, newspaper=43.2
data = pd.DataFrame({'TV': [170.5], 'radio': [13.2], 'newspapaer': [43.2]})
print(round(model2.predict(data)[0], 4))

12.8629


In [10]:
#5-6) 다음의 모델은 통계적 유의미해석에 사용하는 가설에서 모델은 귀무가설을 기각하는가 채택하는가?
# 귀무가설 : 모든 독립변수가 종속변수에 영향을 주지 않는다.
# 대립가설 : 적어도 하나의 독립변수가 종속변수에 영향을 준다.
# 신뢰수준 : 95%
# s = model2.pvalues[1:]
# print("채택" if s.values[1] <= 0.05 else "기각")
# print("채택" if s.values[2] <= 0.05 else "기각")

result = "기각" if model2.f_pvalue <= 0.05 else "채택"
print(result)

기각


In [48]:
# 5-7) 모델에서 가장 영향력 있는 변수의 t-value를 구해, 반올림하여 소수점 아래 3자리까지 출력한다.
s = model2.params[1:].abs().idxmax()
print(round(model2.tvalues[s], 3))

2.125


In [11]:
# 5-8) 모델에서 가장 유의미한 변수의 회귀계수를 구해, 반올림하여 소수점 아래 4자리까지 출력한다.
s = model2.params[1:].abs().idxmin()
print(round(model2.params[s], 4))

0.0011


In [13]:
# 5-9) train 데이터를 사용하여 해당 모델의 예측값과 실제값의 피어슨(pearson) 상관계수를 구하여라.
# 결과는 반올림하여 소수점 아래 3자리까지 출력한다.
# result = model2.predict(train)
# print(result)
temp = pd.DataFrame({'y_true': train['sales'],
                     'y_pred': model2.predict(train)})
result = temp.corr(method='pearson').loc['y_true', 'y_pred']
print(round(result, 3))

0.983


In [14]:
# 5-10) train 데이터를 사용하여 해당 모델의 예측값과 실제값의 스피어만(spearman) 상관계수를 구하여라.
# 결과는 반올림하여 소수점 아래 3자리까지 출력한다.
temp = pd.DataFrame({'y_true': train['sales'], 'y_pred': model2.predict(train)})
result = temp.corr(method='spearman').loc['y_true','y_pred']
print(round(result, 3))

0.994


In [15]:
# 5-11) test 데이터를 사용하여 rmse를 구해, 반올림하여 소수점 아래 4자리까지 출력한다.
from sklearn.metrics import root_mean_squared_error as rmse
result = rmse(test['sales'], model2.predict(test))
print(round(result, 4))

0.8665


In [16]:
# 5-12) train 데이터를 사용하여 잔차를 구하고, 잔차의 IQR을 구해, 반올림하여 소수점 아래 4자리까지 출력한다.
residuals = model2.resid
Q1, Q3 = residuals.quantile([0.25, 0.75])
print(round(Q3 - Q1, 4))

0.9658


In [18]:
# 만일 test 데이터에 대한 잔차를 구하라고 한다면?
residuals = test['sales'] - model2.predict(test)
print(residuals.head(3))

150   -0.623
151    1.122
152    0.386
dtype: float64


In [21]:
#5-13) 통계적으로 가장 유의하지 않은 변수의 표준오차(Standard Error)를 소수점 아래 4자리까지 출력한다.
temp = model2.pvalues[1:].idxmax()
print(round(model2.bse[temp], 4))

0.0104
